In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/WA_Fn-UseC_-Telco-Customer-Churn.csv')

In [ ]:
print(df.shape)
print(df.dtypes)
df.head()

### Типы колонок

`TotalCharges` должна быть числовой (`float`), но загружается как `object`: в части строк
вместо числа стоит пустая строка `" "`. Её нужно привести к числу через
`pd.to_numeric(..., errors='coerce')`, а пропуски заполнить **внутри Pipeline**, не заранее.


In [ ]:
df['Churn'].value_counts().plot(kind='bar')
plt.title('Распределение Churn')
plt.xlabel('Churn')
plt.ylabel('Количество')
plt.show()

print(df['Churn'].value_counts(normalize=True))

### Дисбаланс классов

Примерно **73% не уходят**, **~27% уходят**. Из-за дисбаланса **accuracy бесполезна**:
модель, всегда предсказывающая «не уйдёт», даст ~73% accuracy, не поймав ни одного оттока.
Главная метрика — **recall по классу churn**.


In [ ]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
df[num_cols].hist(bins=30, figsize=(12, 4))
plt.tight_layout()
plt.show()

In [ ]:
# Явные NaN
print(df.isnull().sum())

# Скрытые пустые строки в TotalCharges
print("Пустые строки в TotalCharges:", (df['TotalCharges'].str.strip() == '').sum())

# После конвертации
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print("NaN после конвертации:", df['TotalCharges'].isnull().sum())

### Пропуски

Найдено **11 пустых строк** в `TotalCharges` — после конвертации это `NaN`.
Все 11 — клиенты с `tenure = 0`: только подключились.


In [ ]:
# Contract
print(df.groupby('Contract')['Churn'].value_counts(normalize=True).unstack())

# PaymentMethod
print(df.groupby('PaymentMethod')['Churn'].value_counts(normalize=True).unstack())

# InternetService
print(df.groupby('InternetService')['Churn'].value_counts(normalize=True).unstack())

### Инсайты

- Помесячные контракты уходят в **~43%** против **~3%** у двухлетних.
- **Fiber optic** — **~42%** оттока, максимум среди типов интернета.
- **Электронный чек** связан с наибольшим оттоком (**~45%**).


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, col in zip(axes, ['Contract', 'PaymentMethod', 'InternetService']):
    churn_rate = df.groupby(col)['Churn'].apply(
        lambda x: (x == 'Yes').mean()
    )
    churn_rate.plot(kind='bar', ax=ax)
    ax.set_title(f'Churn rate by {col}')
    ax.set_ylabel('Churn rate')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()